# Fabric Defect Inspector: retrain YOLO at 960 px and check the count

The count diagnosis showed that most missed defects are thin, faint lines and stains that YOLO scores below 0.25, and that changing thresholds helps one roll and hurts the other. So the detector itself has to get better. This notebook:

1. Trains the same YOLO11s on the same data, but at **960 px** instead of 640, so thin defects are 1.5 times wider when the model sees them.
2. Compares old and new on the 140 test photos (mAP50 and recall per class).
3. Runs both on the **test roll** and the **validation roll**, with no threshold tuning, and compares the count.
4. Saves the new weights as `best_960.pt`. Nothing is overwritten.

Training progress is saved to Google Drive. If Colab disconnects, just press Run all again and it continues where it stopped.

**Needs in Google Drive (already there)**
- `My Drive/textile_yolo.zip`
- `My Drive/fabric_inspector_yolo/best.pt`
- `My Drive/fabric_inspector_speed/patchcore_textile_fast.pt`

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all. Takes roughly 60 to 90 minutes, almost all of it training.

## 1. Setup

In [ ]:
!pip install -q ultralytics
import time
import torch, torchvision, numpy as np, cv2, pathlib, shutil, zipfile, yaml, random, collections, os
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU, then run again."
dev = 'cuda'
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive')
ZIP_PATH = DRIVE / 'textile_yolo.zip'
YOLO_PT = DRIVE / 'fabric_inspector_yolo' / 'best.pt'
PC_PT = DRIVE / 'fabric_inspector_speed' / 'patchcore_textile_fast.pt'
for p in (ZIP_PATH, YOLO_PT, PC_PT):
    assert p.exists(), f"Not found in Drive: {p}"
DATA_ROOT = pathlib.Path('/content/data')
OUT = pathlib.Path('/content/retrain_out'); OUT.mkdir(exist_ok=True)

## 2. Load the photos

In [ ]:
random.seed(0); np.random.seed(0)
if ZIP_PATH is not None:
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/')
            if name.endswith('/'):
                continue
            target = DATA_ROOT / name
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)

DS = list(DATA_ROOT.rglob('data.yaml'))[0].parent
NAMES = yaml.safe_load(open(DS / 'data.yaml'))['names']
NAMES = list(NAMES.values()) if isinstance(NAMES, dict) else list(NAMES)

def read_labels(img_path, W, H):
    """Ground-truth boxes for one image: [(class name, x1, y1, x2, y2), ...] in pixels."""
    lp = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    out = []
    if lp.exists():
        for v in (l.split() for l in open(lp) if l.strip()):
            c, cx, cy, w, h = int(v[0]), *map(float, v[1:5])
            out.append((NAMES[c], (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H))
    return out

test_imgs = sorted((DS / 'test' / 'images').glob('*'))
valid_imgs = sorted((DS / 'valid' / 'images').glob('*'))
print(f"classes {NAMES}   test photos {len(test_imgs)}   validation photos {len(valid_imgs)}")

## 3. Train at 960 px
Same model, data, seed and number of epochs as the first training. Only the image size (640 to 960) and the batch size (16 to 8, to fit the GPU) change.

In [ ]:
from ultralytics import YOLO
DATA_YAML = DS / 'data.yaml'
cfg = yaml.safe_load(open(DATA_YAML))
cfg['path'] = str(DS); cfg['train'], cfg['val'], cfg['test'] = 'train/images', 'valid/images', 'test/images'
yaml.safe_dump(cfg, open(DATA_YAML, 'w'), sort_keys=False)

RUN = DRIVE / 'fabric_inspector_yolo960' / 'train'
BEST, LAST = RUN / 'weights' / 'best.pt', RUN / 'weights' / 'last.pt'
if BEST.exists() and (RUN / 'results.png').exists():
    print("training already finished, using", BEST)
elif LAST.exists():
    print("continuing an interrupted training")
    YOLO(str(LAST)).train(resume=True)
else:
    YOLO('yolo11s.pt').train(data=str(DATA_YAML), epochs=60, imgsz=960, batch=8, patience=15,
                             project=str(RUN.parent), name='train', exist_ok=True, seed=0)
NEW_PT = DRIVE / 'fabric_inspector_yolo960' / 'best_960.pt'
shutil.copy(BEST, NEW_PT)
print("new weights:", NEW_PT)

## 4. Old and new on the 140 test photos
The old model is tested at 640 px, the new one at 960 px, each the way it was trained.

In [ ]:
def test_metrics(weights, imgsz, tag):
    m = YOLO(str(weights)).val(data=str(DATA_YAML), split='test', imgsz=imgsz, project='/content/runs', name=tag, exist_ok=True, verbose=False)
    rows = {'all': (m.box.map50, m.box.mr, m.box.mp)}
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, _ = m.box.class_result(i)
        rows[m.names[int(c)]] = (ap50, r, p)
    return rows

old_m, new_m = test_metrics(YOLO_PT, 640, 'old_640'), test_metrics(NEW_PT, 960, 'new_960')
photo_table = pd.DataFrame([{'class': k, 'mAP50 old (640)': old_m[k][0], 'mAP50 new (960)': new_m[k][0],
                             'recall old': old_m[k][1], 'recall new': new_m[k][1],
                             'precision old': old_m[k][2], 'precision new': new_m[k][2]} for k in old_m]).round(3)
photo_table.to_csv(OUT / 'test_photos_old_vs_new.csv', index=False)
photo_table

## 5. Load the models for the roll test
On the roll, the old model sees the frame at its real size. The new model sees it enlarged 1.5 times, which matches how it was trained.

In [ ]:
MODELS = {'old': (YOLO(str(YOLO_PT)), 1.0), 'new': (YOLO(str(NEW_PT)), 1.5)}
LOW_CONF = 0.05

ck = torch.load(PC_PT, map_location='cpu')
bank = ck['bank'].float().to(dev).contiguous(); bank_sq = (bank * bank).sum(1)
PIX_THR = float(ck['pix_thr'])
net = torchvision.models.wide_resnet50_2(weights=torchvision.models.Wide_ResNet50_2_Weights.IMAGENET1K_V1).to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

def yolo_detect(bgr, key='old'):
    """[(class name, confidence, x1, y1, x2, y2), ...] in pixels of the original frame."""
    model, scale = MODELS[key]
    size = int(round(max(bgr.shape[:2]) * scale / 32) * 32)
    r = model.predict(bgr, imgsz=size, conf=LOW_CONF, verbose=False)[0]
    return [(model.names[int(c)], float(p), *map(float, b))
            for b, p, c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist())]

@torch.no_grad()
def anomaly_map(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x = torch.from_numpy(rgb.astype(np.float32) / 255).permute(2, 0, 1)[None].to(dev)
    x = (x - MEAN) / STD
    with torch.autocast('cuda', dtype=torch.float16):
        x = net.maxpool(net.relu(net.bn1(net.conv1(x))))
        l2 = net.layer2(net.layer1(x)); l3 = net.layer3(l2)
        f2 = F.avg_pool2d(l2, 3, 1, 1)
        f3 = F.interpolate(F.avg_pool2d(l3, 3, 1, 1), size=f2.shape[-2:], mode='bilinear', align_corners=False)
        f = torch.cat([f2, f3], 1)[0]
    f = f.float(); C, h, w = f.shape
    q = f.permute(1, 2, 0).reshape(-1, C)
    d2 = torch.cat([((q[i:i + 4096] ** 2).sum(1, keepdim=True) + bank_sq[None] - 2 * (q[i:i + 4096] @ bank.T)).min(1).values
                    for i in range(0, q.shape[0], 4096)])
    return d2.clamp_min(0).sqrt().view(h, w).cpu().numpy()

# speed of each YOLO on one roll-sized frame
_f = np.zeros((640, 960, 3), np.uint8)
for key in MODELS:
    for _ in range(3): yolo_detect(_f, key)
    torch.cuda.synchronize(); t = time.perf_counter()
    for _ in range(20): yolo_detect(_f, key)
    torch.cuda.synchronize(); print(f"YOLO {key}: {(time.perf_counter() - t) / 20 * 1000:.0f} ms per frame")

## 6. Fusion logic

In [ ]:
CELL = 8            # one anomaly-map cell covers 8 x 8 px
MIN_CELLS = 4       # smallest region that counts
COLORS = {'Hole': (60, 60, 240), 'Knot': (0, 145, 255), 'Line': (255, 200, 0), 'Stain': (216, 50, 216), 'unknown': (0, 220, 255)}  # BGR

def smooth(amap):
    return cv2.GaussianBlur(amap.astype(np.float32), (0, 0), 1.0)

def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def overlaps(a, b):
    return min(a[2], b[2]) > max(a[0], b[0]) and min(a[3], b[3]) > max(a[1], b[1])

def centre_in(a, b):
    cx, cy = (a[0] + a[2]) / 2, (a[1] + a[3]) / 2
    return b[0] <= cx <= b[2] and b[1] <= cy <= b[3]

def same_defect(a, b):
    return iou(a, b) >= 0.2 or centre_in(a, b) or centre_in(b, a)

def regions(amap_s, thr):
    """Anomalous regions in a smoothed map: [(score, x1, y1, x2, y2), ...] in pixels, plus the cell mask."""
    mask = (amap_s > thr).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = []
    for i in range(1, n):
        x, y, w, h, a = st[i]
        if a >= MIN_CELLS:
            out.append((float(amap_s[lab == i].max()), x * CELL, y * CELL, (x + w) * CELL, (y + h) * CELL))
    return out, mask

def inspect(bgr, thr, ignore_cols=None):
    """Fused result for one image or frame."""
    known = yolo_detect(bgr)
    amap_s = smooth(anomaly_map(bgr))
    if ignore_cols is not None and len(ignore_cols):
        amap_s[:, ignore_cols] = 0
    regs, mask = regions(amap_s, thr)
    unknown = [r for r in regs if not any(overlaps(r[1:], k[2:]) for k in known)]
    return known, unknown, amap_s, mask

## 7. Build the two rolls and run the models

In [ ]:
N_IMAGES, VIEW_W, STEP = 40, 960, 32
M_PER_IMAGE = 0.15
SCALES = ('old', 'new')

def build_roll(imgs, seed=3):
    random.seed(seed)
    picked = random.sample(imgs, min(N_IMAGES, len(imgs)))
    tiles, gt, seams, x0 = [], [], [], 0
    for i, p in enumerate(picked):
        im = cv2.imread(str(p)); H, W = im.shape[:2]
        if H != 640:
            im = cv2.resize(im, (int(W * 640 / H), 640)); H, W = im.shape[:2]
        for cls, x1, y1, x2, y2 in read_labels(p, W, H):
            gt.append({'class': cls, 'box': (x0 + x1, y1, x0 + x2, y2), 'photo': i, 'local': (x1, y1, x2, y2)})
        tiles.append(im); x0 += W; seams.append(x0)
    roll = np.concatenate(tiles, axis=1)
    return {'roll': roll, 'tiles': tiles, 'gt': gt, 'seams': seams, 'len': roll.shape[1],
            'offsets': list(range(0, roll.shape[1] - VIEW_W + 1, STEP)), 'px_per_m': (roll.shape[1] / len(tiles)) / M_PER_IMAGE}

def run_models(R):
    R['yolo'] = {s: [] for s in SCALES}; R['regs'] = []
    for off in R['offsets']:
        frame = np.ascontiguousarray(R['roll'][:, off:off + VIEW_W])
        for s in SCALES:
            R['yolo'][s].append(yolo_detect(frame, s))
        seam_x = [s - off for s in R['seams'] if 0 < s - off < VIEW_W]
        ignore = sorted({c for s in seam_x for c in range(max(s // CELL - 5, 0), min(s // CELL + 5, VIEW_W // CELL))})
        a = smooth(anomaly_map(frame))
        if ignore:
            a[:, ignore] = 0
        R['regs'].append(regions(a, PIX_THR)[0])
    # YOLO on every photo by itself, as in the single-photo test
    R['solo'] = []

ROLLS = {'test': build_roll(test_imgs), 'validation': build_roll(valid_imgs)}
for name, R in ROLLS.items():
    t0 = time.time(); run_models(R)
    print(f"{name} roll: {len(R['tiles'])} photos, {R['len']} px, {len(R['gt'])} labelled defects, {len(R['offsets'])} frames, models run in {time.time() - t0:.0f} s")

## 8. Count with the old and the new model
Same settings for both: confidence 0.25, a defect must be seen in 2 frames. The last two rows add the two rolls together.

In [ ]:
def same_track(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    if inter <= 0:
        return False
    area_a, area_b = (a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter) >= 0.2 or inter / min(area_a, area_b) >= 0.5

def replay(R, scale='old', conf=0.25, conf_small=None, seam_filter=True, min_hits=2):
    """Count the roll from the stored model results. conf_small, if given, is the confidence used for Knot and Line."""
    need = lambda cls: conf_small if (conf_small is not None and cls in ('Knot', 'Line')) else conf
    tracks = []
    def add(kind, cls, c, box):
        for t in tracks:
            if same_track(box, t['box']) or (kind != t['kind'] and overlaps(box, t['box'])):
                t['hits'] += 1
                if kind == 'known':
                    if t['kind'] == 'unknown' or c > t['best']:
                        t['box'], t['best'] = list(box), c
                    t['kind'] = 'known'; t['votes'][cls] = t['votes'].get(cls, 0) + c
                return
        tracks.append({'box': list(box), 'best': c if kind == 'known' else 0.0,
                       'votes': {cls: c} if kind == 'known' else {}, 'hits': 1, 'kind': kind})
    for n, off in enumerate(R['offsets']):
        seam_x = [s - off for s in R['seams'] if 0 < s - off < VIEW_W]
        known = [k for k in R['yolo'][scale][n] if k[1] >= need(k[0])]
        if seam_filter:
            known = [k for k in known if not (k[4] - k[2] <= 64 and any(k[2] - 8 < sx < k[4] + 8 for sx in seam_x))]
        unknown = [r for r in R['regs'][n] if not any(overlaps(r[1:], k[2:]) for k in known)]
        for cls, c, x1, y1, x2, y2 in known:
            if x1 > 2 and x2 < VIEW_W - 2:
                add('known', cls, c, (off + x1, y1, off + x2, y2))
        for score, x1, y1, x2, y2 in unknown:
            if x1 > 2 and x2 < VIEW_W - 2:
                add('unknown', None, score, (off + x1, y1, off + x2, y2))
    final = [t for t in tracks if t['hits'] >= min_hits]
    found = [any(overlaps(g['box'], t['box']) for t in final) for g in R['gt']]
    real = [any(overlaps(t['box'], g['box']) for g in R['gt']) for t in final]
    n_gt = len(R['gt'])
    return {'labelled defects': n_gt, 'defects counted': len(final), 'count error %': round((len(final) - n_gt) / n_gt * 100, 1),
            'labelled defects found': int(sum(found)), 'share found %': round(sum(found) / n_gt * 100, 1),
            'counted but not labelled': int(len(final) - sum(real)),
            'unknown': int(sum(t['kind'] == 'unknown' for t in final))}, found

base_test, found_test = replay(ROLLS['test'])
print("current settings on the test roll (should be 53 counted, 54 found):", base_test)

In [ ]:
rows = []
for roll_name, R in ROLLS.items():
    for key in MODELS:
        res, _ = replay(R, scale=key)
        rows.append({'roll': roll_name, 'YOLO': key, **res})
t = pd.DataFrame(rows)
for key in MODELS:
    s = t[t.YOLO == key]
    n, c, f = s['labelled defects'].sum(), s['defects counted'].sum(), s['labelled defects found'].sum()
    rows.append({'roll': 'both together', 'YOLO': key, 'labelled defects': n, 'defects counted': c, 'count error %': round((c - n) / n * 100, 1),
                 'labelled defects found': f, 'share found %': round(f / n * 100, 1),
                 'counted but not labelled': s['counted but not labelled'].sum(), 'unknown': s['unknown'].sum()})
roll_table = pd.DataFrame(rows)
roll_table.to_csv(OUT / 'rolls_old_vs_new.csv', index=False)
roll_table

## 9. Save
Tables go to the Drive folder `fabric_inspector_yolo960`, next to `best_960.pt`.

In [ ]:
if DRIVE is not None:
    DEST = DRIVE / 'fabric_inspector_yolo960'; DEST.mkdir(exist_ok=True)
    for f in OUT.iterdir():
        shutil.copy(f, DEST / f.name)
    print("saved to", DEST, sorted(p.name for p in DEST.iterdir() if p.is_file()))